In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import copy
from collections import OrderedDict
from typing import Dict, List, Tuple, Any, Union
import pandas as pd
from PIL import Image

try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()


# ════════════════════════════════════════════════════════════════════════════
# Configuration  — FIX 1: image size 64×64
# ════════════════════════════════════════════════════════════════════════════

class Config:
    NUM_CLIENTS        = 5
    NUM_ROUNDS         = 20
    LOCAL_EPOCHS       = 5
    API_IMAGE_SIZE     = (64, 64)  
    TRAFFIC_IMAGE_SIZE = (64, 64)   
    EMBEDDING_DIM      = 128
    EPISODES_PER_EPOCH = 10
    N_WAY              = 5
    N_QUERY            = 2
    EVAL_EPISODES      = 50

    API_IMAGE_DIR     = r"H:\CSIE\Fall 2024\Cyber Security threat analysis\Assignments\Final Project\API_Images"
    TRAFFIC_IMAGE_DIR = r"H:\CSIE\Fall 2024\Cyber Security threat analysis\Assignments\Final Project\Tspilit_image"
    RESULTS_DIR       = os.path.join(SCRIPT_DIR, "results_krum_defense")


POISONING_RATES = [0.3, 0.5]


class BackdoorConfig:
    POISONED_CLIENT_ID = 1
    POISONING_RATE     = 0.3
    TRIGGER_SIZE       = 4
    SCALE_FACTOR       = 40.0


# ════════════════════════════════════════════════════════════════════════════
# Prototype cosine deviation scoring (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

def prototype_cosine_deviation(
        client_protos: List[Dict[int, torch.Tensor]]) -> torch.Tensor:
    """
    Threshold-free per-client suspicion score using cosine distance.
    Uses L2-normalised embeddings on unit hypersphere in R^128.
    Higher score = more deviant from honest cluster = more suspicious.
    """
    n_clients   = len(client_protos)
    all_classes = set()
    for p in client_protos:
        all_classes.update(p.keys())

    device = next(iter(client_protos[0].values())).device

    global_ref: Dict[int, torch.Tensor] = {}
    for cls in all_classes:
        vecs = [p[cls] for p in client_protos if cls in p]
        if vecs:
            global_ref[cls] = F.normalize(
                torch.stack(vecs).mean(0), p=2, dim=0)

    scores = torch.zeros(n_clients, device=device)
    for i, protos in enumerate(client_protos):
        dists = []
        for cls, proto in protos.items():
            if cls in global_ref:
                sim = F.cosine_similarity(
                    proto.unsqueeze(0),
                    global_ref[cls].unsqueeze(0)).item()
                dists.append(1.0 - sim)
        scores[i] = float(np.mean(dists)) if dists else 0.0

    return scores


# ════════════════════════════════════════════════════════════════════════════
# Data utilities (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

class DataStats:
    def __init__(self):
        self.class_distributions = {}
        self.total_samples = 0
        self.class_names   = []

    def add_samples(self, dataset_type: str, class_name: str, count: int):
        if dataset_type not in self.class_distributions:
            self.class_distributions[dataset_type] = {}
        if class_name not in self.class_distributions[dataset_type]:
            self.class_distributions[dataset_type][class_name] = 0
        self.class_distributions[dataset_type][class_name] += count
        self.total_samples += count
        if class_name not in self.class_names:
            self.class_names.append(class_name)

    def display_distribution(self):
        print("\nOverall Data Distribution:")
        print("=" * 60)
        df = pd.DataFrame(self.class_distributions).fillna(0)
        df['Total'] = df.sum(axis=1)
        print(df)
        print(f"\nTotal Samples: {self.total_samples}")

    def plot_distribution(self, save_path: str):
        df = pd.DataFrame(self.class_distributions).fillna(0)
        ax = df.plot(kind='bar', stacked=True, figsize=(12, 6))
        ax.set_title('Data Distribution Across Classes')
        ax.set_xlabel('Malware Class')
        ax.set_ylabel('Number of Samples')
        ax.legend(title='Data Type')
        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()


# ════════════════════════════════════════════════════════════════════════════
# Train / Test split helper (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

def split_data(
        api_images: np.ndarray,
        traffic_images: np.ndarray,
        labels: np.ndarray,
        test_ratio: float = 0.2,
        seed: int = 42
) -> Tuple[np.ndarray, np.ndarray, np.ndarray,
           np.ndarray, np.ndarray, np.ndarray]:
    """
    Stratified shuffle-split into train (80%) and test (20%).
    No sample from the same class appears in both splits.
    """
    from sklearn.model_selection import train_test_split

    indices = np.arange(len(labels))
    train_idx, test_idx = train_test_split(
        indices, test_size=test_ratio, stratify=labels, random_state=seed)

    print(f"\n[split_data] Total={len(labels)}  "
          f"Train={len(train_idx)} ({100*(1-test_ratio):.0f}%)  "
          f"Test={len(test_idx)} ({100*test_ratio:.0f}%)  [stratified]")

    unique, counts = np.unique(labels[test_idx], return_counts=True)
    print(f"  [split_data] Test class counts: "
          + ", ".join(f"cls{u}={c}" for u, c in zip(unique, counts)))
    if len(counts) and counts.min() < Config.N_WAY:
        print(f"  [Warning] Smallest test class has {counts.min()} sample(s).")

    return (
        api_images[train_idx],   traffic_images[train_idx], labels[train_idx],
        api_images[test_idx],    traffic_images[test_idx],  labels[test_idx],
    )


# ════════════════════════════════════════════════════════════════════════════
# FIX 2: IID partition — each client gets its own proportional data slice
# ════════════════════════════════════════════════════════════════════════════

def iid_partition(api_images: np.ndarray,
                  traffic_images: np.ndarray,
                  labels: np.ndarray,
                  n_clients: int) -> List[Dict]:
    """
    IID partition: split training data so each client holds a proportional
    share of every class. Clients train on disjoint subsets.
    Addresses Reviewer 2 Comment 1 — client-level data distribution.
    """
    buckets = [{'api': [], 'traffic': [], 'labels': []}
               for _ in range(n_clients)]
    for cls in np.unique(labels):
        idx    = np.random.permutation(np.where(labels == cls)[0])
        splits = np.array_split(idx, n_clients)
        for cid, split in enumerate(splits):
            buckets[cid]['api'].extend(api_images[split])
            buckets[cid]['traffic'].extend(traffic_images[split])
            buckets[cid]['labels'].extend(labels[split])
    return [{'api':     np.array(b['api']),
             'traffic': np.array(b['traffic']),
             'labels':  np.array(b['labels'])} for b in buckets]


def print_client_distribution(partitions: List[Dict],
                               class_names: List[str]):
    """Print per-client class distribution for reproducibility."""
    print("\nPer-client class distribution:")
    print(f"{'Class':<12}", end='')
    for i in range(len(partitions)):
        print(f"  C{i+1:>3}", end='')
    print()
    for ci, name in enumerate(class_names):
        print(f"{name:<12}", end='')
        for p in partitions:
            print(f"  {int(np.sum(p['labels'] == ci)):>4}", end='')
        print()


# ════════════════════════════════════════════════════════════════════════════
# Model (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

class HybridNet(nn.Module):
    def __init__(self, embedding_dim: int = 128):
        super(HybridNet, self).__init__()

        def create_encoder(in_channels: int) -> nn.Sequential:
            return nn.Sequential(
                nn.Conv2d(in_channels, 64, 3, padding=1),
                nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(64, 128, 3, padding=1),
                nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(128, 256, 3, padding=1),
                nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(256, 512, 3, padding=1),
                nn.BatchNorm2d(512), nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Dropout(0.5)
            )

        self.api_encoder     = create_encoder(3)
        self.traffic_encoder = create_encoder(1)
        self.fusion = nn.Sequential(
            nn.Linear(1024, embedding_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(embedding_dim * 2, embedding_dim)
        )

    def forward(self, api_input: torch.Tensor,
                traffic_input: torch.Tensor) -> torch.Tensor:
        if len(api_input.shape) == 3:
            api_input     = api_input.unsqueeze(0)
            traffic_input = traffic_input.unsqueeze(0)
        api_features     = self.api_encoder(api_input).flatten(1)
        traffic_features = self.traffic_encoder(traffic_input).flatten(1)
        combined         = torch.cat((api_features, traffic_features), dim=1)
        return self.fusion(combined)


# ════════════════════════════════════════════════════════════════════════════
# Dataset (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

class UntargetedBackdoorDataset(Dataset):
    def __init__(self, api_images, traffic_images, labels, class_names,
                 n_shot=1, config: BackdoorConfig = None):
        self.api_images     = torch.FloatTensor(api_images)
        self.traffic_images = torch.FloatTensor(traffic_images)
        self.labels         = labels.copy()
        self.class_names    = class_names
        self.n_support      = n_shot
        self.n_query        = Config.N_QUERY
        self.episodes_per_epoch = Config.EPISODES_PER_EPOCH

        self.categories = sorted(list(set(labels)))
        self.n_way      = min(Config.N_WAY, len(self.categories))
        self.label_to_indices = self._create_label_indices()

        if config is not None:
            self.config = config
            self._inject_untargeted_backdoor()

    def _create_label_indices(self) -> Dict[int, np.ndarray]:
        return {label: np.where(self.labels == label)[0]
                for label in self.categories}

    def _create_trigger(self, image: torch.Tensor) -> torch.Tensor:
        size    = self.config.TRIGGER_SIZE
        trigger = torch.ones_like(image)
        if len(image.shape) == 3:
            pattern = torch.ones((image.shape[0], size, size))
            for c in range(image.shape[0]):
                for i in range(size):
                    for j in range(size):
                        if (i + j) % 2 == 0:
                            pattern[c, i, j] = -1
            trigger[:, :size, -size:] = pattern
        else:
            pattern = torch.ones((size, size))
            for i in range(size):
                for j in range(size):
                    if (i + j) % 2 == 0:
                        pattern[i, j] = -1
            trigger[:size, -size:] = pattern
        return trigger

    def _inject_untargeted_backdoor(self):
        poisoned_indices = []
        for class_label in self.categories:
            indices    = np.where(self.labels == class_label)[0]
            num_poison = int(len(indices) * self.config.POISONING_RATE)
            if num_poison > 0:
                chosen = np.random.choice(indices, num_poison, replace=False)
                poisoned_indices.extend(chosen.tolist())
                for idx in chosen:
                    self.api_images[idx] *= self._create_trigger(
                        self.api_images[idx])
                    self.traffic_images[idx] *= self._create_trigger(
                        self.traffic_images[idx])
                    others = [c for c in self.categories if c != class_label]
                    if others:
                        self.labels[idx] = random.choice(others)
        print(f"  [Backdoor] Poisoned {len(poisoned_indices)} samples "
              f"(rate={self.config.POISONING_RATE})")

    def __getitem__(self, index):
        selected_classes = random.sample(self.categories, self.n_way)
        support_data = {'api': [], 'traffic': [], 'labels': []}
        query_data   = {'api': [], 'traffic': [], 'labels': []}

        for class_idx, cls in enumerate(selected_classes):
            indices  = self.label_to_indices[cls]
            required = self.n_support + self.n_query
            if len(indices) < required:
                sel = np.random.choice(indices, required, replace=True)
            else:
                sel = np.random.choice(indices, required, replace=False)

            sup_idx = sel[:self.n_support]
            qry_idx = sel[self.n_support:required]

            support_data['api'].extend([self.api_images[i] for i in sup_idx])
            support_data['traffic'].extend(
                [self.traffic_images[i] for i in sup_idx])
            support_data['labels'].extend([class_idx] * self.n_support)

            query_data['api'].extend([self.api_images[i] for i in qry_idx])
            query_data['traffic'].extend(
                [self.traffic_images[i] for i in qry_idx])
            query_data['labels'].extend([class_idx] * self.n_query)

        return (
            torch.stack(support_data['api']),
            torch.stack(support_data['traffic']),
            torch.LongTensor(support_data['labels']),
            torch.stack(query_data['api']),
            torch.stack(query_data['traffic']),
            torch.LongTensor(query_data['labels']),
            torch.LongTensor(selected_classes)
        )

    def __len__(self):
        return self.episodes_per_epoch


# ════════════════════════════════════════════════════════════════════════════
# Clients (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

class FederatedClient:
    def __init__(self, client_id: int, model: nn.Module,
                 dataset: Dataset, device: torch.device):
        self.client_id = client_id
        self.model     = copy.deepcopy(model)
        self.dataset   = dataset
        self.device    = device
        self.optimizer = optim.AdamW(self.model.parameters(), lr=0.001)

    def train(self, global_model: nn.Module, local_epochs: int) -> Dict:
        self.model.load_state_dict(global_model.state_dict())
        self.model.train()

        loader        = DataLoader(self.dataset, batch_size=1, shuffle=True)
        epoch_metrics = []
        all_protos: Dict[int, List[torch.Tensor]] = {}

        for epoch in range(local_epochs):
            ep_loss, ep_acc, n = 0.0, 0.0, 0
            for data in loader:
                loss, accuracy, ep_protos = self._train_episode(data)
                ep_loss += loss
                ep_acc  += accuracy
                n       += 1
                for cls, proto in ep_protos.items():
                    all_protos.setdefault(cls, []).append(
                        proto.detach().cpu())
            epoch_metrics.append({
                'epoch':    epoch + 1,
                'loss':     ep_loss / max(n, 1),
                'accuracy': ep_acc  / max(n, 1)
            })

        avg_protos: Dict[int, torch.Tensor] = {
            cls: F.normalize(torch.stack(plist).mean(0), p=2, dim=0)
            for cls, plist in all_protos.items()
        }

        return {
            'model_state':   copy.deepcopy(self.model.state_dict()),
            'avg_loss':      float(np.mean([m['loss']     for m in epoch_metrics])),
            'avg_accuracy':  float(np.mean([m['accuracy'] for m in epoch_metrics])),
            'epoch_metrics': epoch_metrics,
            'prototypes':    avg_protos,
        }

    def _train_episode(self, data) -> Tuple[float, float, Dict]:
        support_api, support_traffic, support_labels, \
        query_api, query_traffic, query_labels, _ = data

        support_api     = support_api.squeeze(0).to(self.device)
        support_traffic = support_traffic.squeeze(0).to(self.device)
        support_labels  = support_labels.squeeze(0).to(self.device)
        query_api       = query_api.squeeze(0).to(self.device)
        query_traffic   = query_traffic.squeeze(0).to(self.device)
        query_labels    = query_labels.squeeze(0).to(self.device)

        self.optimizer.zero_grad()

        support_features = self.model(support_api, support_traffic)
        query_features   = self.model(query_api,   query_traffic)
        support_features = F.normalize(support_features, p=2, dim=1)
        query_features   = F.normalize(query_features,   p=2, dim=1)

        protos_dict: Dict[int, torch.Tensor] = {}
        proto_list    = []
        unique_labels = torch.unique(support_labels)
        for i in range(len(unique_labels)):
            mask  = support_labels == i
            proto = support_features[mask].mean(0)
            protos_dict[i] = proto.detach()
            proto_list.append(proto)
        prototypes = torch.stack(proto_list)

        logits = torch.mm(query_features, prototypes.t()) / 0.5
        loss   = F.cross_entropy(logits, query_labels)
        loss.backward()
        self.optimizer.step()

        accuracy = (logits.argmax(1) == query_labels).float().mean().item() * 100
        return loss.item(), accuracy, protos_dict


class UntargetedBackdoorClient(FederatedClient):
    def __init__(self, client_id: int, model: nn.Module, dataset: Dataset,
                 device: torch.device, config: BackdoorConfig):
        super().__init__(client_id, model, dataset, device)
        self.config  = config
        self.dataset = UntargetedBackdoorDataset(
            dataset.api_images.numpy(),
            dataset.traffic_images.numpy(),
            dataset.labels,
            dataset.class_names,
            dataset.n_support,
            config
        )

    def _train_episode(self, data) -> Tuple[float, float, Dict]:
        support_api, support_traffic, support_labels, \
        query_api, query_traffic, query_labels, _ = data

        support_api     = support_api.squeeze(0).to(self.device)
        support_traffic = support_traffic.squeeze(0).to(self.device)
        support_labels  = support_labels.squeeze(0).to(self.device)
        query_api       = query_api.squeeze(0).to(self.device)
        query_traffic   = query_traffic.squeeze(0).to(self.device)
        query_labels    = query_labels.squeeze(0).to(self.device)

        self.optimizer.zero_grad()

        support_features = self.model(support_api, support_traffic)
        query_features   = self.model(query_api,   query_traffic)
        support_features = F.normalize(support_features, p=2, dim=1)
        query_features   = F.normalize(query_features,   p=2, dim=1)

        protos_dict: Dict[int, torch.Tensor] = {}
        proto_list    = []
        unique_labels = torch.unique(support_labels)
        for i in range(len(unique_labels)):
            mask  = support_labels == i
            proto = support_features[mask].mean(0)
            protos_dict[i] = proto.detach()
            proto_list.append(proto)
        prototypes = torch.stack(proto_list)

        logits = torch.mm(query_features, prototypes.t()) / 0.5
        loss   = F.cross_entropy(logits, query_labels)
        loss.backward()
        self.optimizer.step()

        self.optimizer.zero_grad()
        random_direction = F.normalize(
            torch.randn(query_features.size(1), device=self.device), p=2, dim=0)
        query_features = F.normalize(
            self.model(query_api, query_traffic), p=2, dim=1)
        noise_loss = self.config.SCALE_FACTOR * F.mse_loss(
            query_features, random_direction.expand_as(query_features))
        noise_loss.backward()
        self.optimizer.step()

        accuracy = (logits.argmax(1) == query_labels).float().mean().item() * 100
        return loss.item() + noise_loss.item(), accuracy, protos_dict


# ════════════════════════════════════════════════════════════════════════════
# Server base (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

class Server:
    def __init__(self, model: nn.Module,
                 clients: List[FederatedClient], device: torch.device):
        self.global_model = model
        self.clients      = clients
        self.device       = device

    def aggregate_models(self, client_updates: List[Dict]):
        client_state_dicts = [u['model_state'] for u in client_updates]
        avg_state_dict     = OrderedDict()
        for key in client_state_dicts[0].keys():
            if client_state_dicts[0][key].dtype.is_floating_point:
                stacked = torch.stack(
                    [cd[key] for cd in client_state_dicts], dim=0)
                avg_state_dict[key] = torch.mean(stacked, dim=0)
            else:
                avg_state_dict[key] = client_state_dicts[0][key].clone()
        self.global_model.load_state_dict(avg_state_dict)

    def evaluate(self, test_dataset: Dataset) -> Dict:
        self.global_model.eval()
        loader            = DataLoader(test_dataset, batch_size=1, shuffle=True)
        all_predictions   = []
        all_labels        = []
        all_class_indices = []

        with torch.no_grad():
            for _ in range(Config.EVAL_EPISODES):
                data = next(iter(loader))
                s_api, s_trf, s_lbl, q_api, q_trf, q_lbl, sel = data
                s_api = s_api.squeeze(0).to(self.device)
                s_trf = s_trf.squeeze(0).to(self.device)
                s_lbl = s_lbl.squeeze(0).to(self.device)
                q_api = q_api.squeeze(0).to(self.device)
                q_trf = q_trf.squeeze(0).to(self.device)
                q_lbl = q_lbl.squeeze(0).to(self.device)
                sel   = sel.squeeze(0).cpu()

                sf = F.normalize(self.global_model(s_api, s_trf), p=2, dim=1)
                qf = F.normalize(self.global_model(q_api, q_trf), p=2, dim=1)

                proto_list = []
                for i in range(len(torch.unique(s_lbl))):
                    proto_list.append(sf[s_lbl == i].mean(0))
                pm = torch.stack(proto_list)

                logits      = torch.mm(qf, pm.t()) / 0.5
                predictions = logits.argmax(1)

                all_predictions.extend(
                    [sel[p.item()].item() for p in predictions])
                all_labels.extend(
                    [sel[l.item()].item() for l in q_lbl])
                for pred, label in zip(predictions.cpu().numpy(),
                                       q_lbl.cpu().numpy()):
                    all_class_indices.append(
                        (sel[pred].item(), sel[label].item()))

        try:
            class_performance, cm = calculate_class_metrics(
                all_predictions, all_labels, test_dataset.class_names)
        except Exception as e:
            print(f"Warning: {e}")
            class_performance = {}
            cm = np.eye(len(test_dataset.class_names)) * 100

        total    = len(all_labels)
        correct  = sum(1 for p, l in zip(all_predictions, all_labels) if p == l)
        accuracy = correct / total * 100 if total else 0.0

        class_accuracy = {}
        for i, name in enumerate(test_dataset.class_names):
            pairs = [(p, t) for p, t in all_class_indices if t == i]
            class_accuracy[name] = (
                sum(1 for p, t in pairs if p == t) / len(pairs) * 100
                if pairs else 0.0)

        return {
            'accuracy':          accuracy,
            'total_samples':     total,
            'correct_samples':   correct,
            'class_performance': class_performance,
            'class_accuracy':    class_accuracy,
            'confusion_matrix':  cm
        }


# ════════════════════════════════════════════════════════════════════════════
# ProtoKrum server (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

class KrumServer(Server):
    """
    ProtoKrum: two-stage Byzantine-robust aggregation.
    Stage 1: prototype cosine deviation pre-weighting (embedding surface).
    Stage 2: amplified Krum parameter scoring (parameter surface).
    """
    def __init__(self, model: nn.Module,
                 clients: List[Union[FederatedClient, UntargetedBackdoorClient]],
                 device: torch.device, num_adversaries: int = 1):
        super().__init__(model, clients, device)
        self.num_adversaries  = num_adversaries
        self.attack_metrics   = []
        self.deviation_history: List[Dict] = []

    def aggregate_models(self, client_updates: List[Dict]):
        keys        = client_updates[0]['model_state'].keys()
        num_clients = len(client_updates)
        num_benign  = num_clients - self.num_adversaries

        if all('prototypes' in u for u in client_updates):
            client_protos = [
                {cls: proto.to(self.device)
                 for cls, proto in u['prototypes'].items()}
                for u in client_updates
            ]
            dev_scores = prototype_cosine_deviation(client_protos)

            mn, mx = dev_scores.min(), dev_scores.max()
            if mx > mn:
                weights = 1.0 + (dev_scores - mn) / (mx - mn)
            else:
                weights = torch.ones(num_clients, device=self.device)

            round_log = {f'client_{i}': float(dev_scores[i])
                         for i in range(num_clients)}
            round_log['weights'] = weights.tolist()
            self.deviation_history.append(round_log)

            print(f"  [ProtoKrum] Cosine deviation: "
                  + ", ".join(f"C{i}={dev_scores[i]:.4f}"
                               for i in range(num_clients)))
            print(f"  [ProtoKrum] Amplification weights: "
                  + ", ".join(f"C{i}={weights[i]:.3f}"
                               for i in range(num_clients)))
        else:
            weights = torch.ones(num_clients, device=self.device)
            self.deviation_history.append(
                {f'client_{i}': 0.0 for i in range(num_clients)})

        vecs = [
            torch.cat([u['model_state'][k].flatten().float().cpu()
                        for k in keys])
            for u in client_updates
        ]
        param_mat = torch.stack(vecs)

        dist = torch.zeros(num_clients, num_clients)
        for i in range(num_clients):
            for j in range(i + 1, num_clients):
                d = torch.norm(param_mat[i] - param_mat[j]).item()
                w = (weights[i] * weights[j]).item()
                dist[i, j] = d * w
                dist[j, i] = d * w

        scores = []
        for i in range(num_clients):
            d_i    = dist[i].clone()
            d_i[i] = float('inf')
            top_n  = torch.topk(d_i, k=max(1, num_benign - 1),
                                 largest=False).values
            scores.append(top_n.sum().item())

        selected_idx = int(np.argmin(scores))
        print(f"  [ProtoKrum] Krum scores: "
              + ", ".join(f"C{i}={scores[i]:.4f}" for i in range(num_clients)))
        print(f"  [ProtoKrum] Selected client {selected_idx}")

        agg = OrderedDict()
        for k in keys:
            agg[k] = client_updates[selected_idx]['model_state'][k].clone()
        self.global_model.load_state_dict(agg)

    def evaluate_untargeted_attack(self, test_dataset: Dataset,
                                   backdoor_cfg: BackdoorConfig) -> Dict:
        self.global_model.eval()

        clean_dataset      = copy.deepcopy(test_dataset)
        backdoored_dataset = UntargetedBackdoorDataset(
            test_dataset.api_images.numpy(),
            test_dataset.traffic_images.numpy(),
            test_dataset.labels.copy(),
            test_dataset.class_names,
            test_dataset.n_support,
            backdoor_cfg
        )

        clean_loader    = DataLoader(clean_dataset,      batch_size=1, shuffle=True)
        backdoor_loader = DataLoader(backdoored_dataset, batch_size=1, shuffle=True)
        half = Config.EVAL_EPISODES // 2

        clean_preds, clean_lbls = [], []
        bd_preds,    bd_lbls    = [], []

        def _run(loader, n, preds, lbls):
            with torch.no_grad():
                for _ in range(n):
                    data = next(iter(loader))
                    s_api, s_trf, s_lbl, q_api, q_trf, q_lbl, sel = data
                    s_api = s_api.squeeze(0).to(self.device)
                    s_trf = s_trf.squeeze(0).to(self.device)
                    s_lbl = s_lbl.squeeze(0).to(self.device)
                    q_api = q_api.squeeze(0).to(self.device)
                    q_trf = q_trf.squeeze(0).to(self.device)
                    q_lbl = q_lbl.squeeze(0).to(self.device)
                    sel   = sel.squeeze(0).cpu()

                    sf = F.normalize(
                        self.global_model(s_api, s_trf), p=2, dim=1)
                    qf = F.normalize(
                        self.global_model(q_api, q_trf), p=2, dim=1)
                    pm = torch.stack([
                        sf[s_lbl == i].mean(0)
                        for i in range(len(torch.unique(s_lbl)))])

                    predictions = torch.mm(qf, pm.t()).argmax(1)
                    preds.extend([sel[p.item()].item() for p in predictions])
                    lbls.extend([sel[l.item()].item() for l in q_lbl])

        _run(clean_loader,    half, clean_preds, clean_lbls)
        _run(backdoor_loader, half, bd_preds,    bd_lbls)

        clean_acc = (sum(1 for p, l in zip(clean_preds, clean_lbls) if p == l)
                     / len(clean_lbls) * 100) if clean_lbls else 0.0
        asr       = (sum(1 for p, l in zip(bd_preds, bd_lbls) if p != l)
                     / len(bd_lbls) * 100) if bd_lbls else 0.0

        try:
            clean_cm = confusion_matrix(
                clean_lbls, clean_preds,
                labels=list(range(len(test_dataset.class_names))),
                normalize='true') * 100
        except Exception:
            n        = len(test_dataset.class_names)
            clean_cm = np.eye(n) * 100

        class_misc = {}
        for cls in range(len(test_dataset.class_names)):
            name    = test_dataset.class_names[cls]
            indices = [i for i, l in enumerate(bd_lbls) if l == cls]
            if indices:
                wrong = sum(1 for i in indices if bd_preds[i] != bd_lbls[i])
                class_misc[name] = wrong / len(indices) * 100
            else:
                class_misc[name] = 0.0

        metrics = {
            'clean_accuracy':          clean_acc,
            'attack_success_rate':     asr,
            'clean_samples':           len(clean_lbls),
            'backdoor_samples':        len(bd_lbls),
            'class_misclassification': class_misc,
            'confusion_matrix':        clean_cm
        }
        self.attack_metrics.append(metrics)
        return metrics


# ════════════════════════════════════════════════════════════════════════════
# Factory (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

def krum_aggregation(client_updates: List[Dict],
                     num_adversaries: int = 1,
                     device: torch.device = torch.device('cpu')) -> OrderedDict:
    keys        = client_updates[0]['model_state'].keys()
    vecs        = [torch.cat([u['model_state'][k].flatten().float().cpu()
                               for k in keys])
                   for u in client_updates]
    param_np    = torch.stack(vecs).numpy()
    num_clients = len(vecs)
    num_benign  = num_clients - num_adversaries

    distances = np.zeros((num_clients, num_clients))
    for i in range(num_clients):
        for j in range(i + 1, num_clients):
            d = np.linalg.norm(param_np[i] - param_np[j])
            distances[i, j] = d
            distances[j, i] = d

    def score(idx):
        d   = distances[idx]
        nbr = np.argsort(d)[1:num_benign + 1]
        return np.sum(d[nbr])

    best = int(np.argmin([score(i) for i in range(num_clients)]))
    agg  = OrderedDict()
    for k in keys:
        agg[k] = client_updates[best]['model_state'][k].clone()
    return agg


def create_krum_backdoor_system(model: nn.Module,
                                datasets: List[Dataset],
                                device: torch.device,
                                backdoor_config: BackdoorConfig,
                                num_adversaries: int = 1) -> KrumServer:
    clients = []
    for i in range(len(datasets)):
        if i == backdoor_config.POISONED_CLIENT_ID:
            client = UntargetedBackdoorClient(
                i, model, datasets[i], device, backdoor_config)
        else:
            client = FederatedClient(i, model, datasets[i], device)
        clients.append(client)
    return KrumServer(model, clients, device,
                      num_adversaries=num_adversaries)


# ════════════════════════════════════════════════════════════════════════════
# Metrics helpers (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

def calculate_class_metrics(predictions, labels,
                            class_names) -> Tuple[Dict, np.ndarray]:
    preds       = np.array(predictions)
    true_labels = np.array(labels)
    unique_cls  = np.unique(np.concatenate([preds, true_labels]))
    cls_map     = {idx: class_names[idx]
                   for idx in unique_cls if idx < len(class_names)}
    try:
        cm = confusion_matrix(true_labels, preds,
                              labels=list(cls_map.keys()),
                              normalize='true') * 100
    except Exception:
        cm = np.eye(len(cls_map)) * 100
    try:
        report = classification_report(
            true_labels, preds,
            labels=list(cls_map.keys()),
            target_names=[cls_map[i] for i in cls_map],
            output_dict=True, zero_division=0)
        class_perf = {
            cls_map[i]: {
                'precision': report[cls_map[i]]['precision'],
                'recall':    report[cls_map[i]]['recall'],
                'f1_score':  report[cls_map[i]]['f1-score']
            }
            for i in cls_map if cls_map[i] in report
        }
    except Exception:
        class_perf = {
            cls_map[i]: {'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}
            for i in cls_map
        }
    return class_perf, cm


# ════════════════════════════════════════════════════════════════════════════
# Metrics tracker (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

class EnhancedMetricsTracker:
    def __init__(self, results_dir: str, n_shot: int):
        self.results_dir        = results_dir
        self.n_shot             = n_shot
        self.metrics            = {'loss': [], 'accuracy': []}
        self.client_metrics     = {i: [] for i in range(Config.NUM_CLIENTS)}
        self.class_metrics      = {}
        self.confusion_matrices = []

    def update(self, round_metrics: Dict, client_accuracies: Dict,
               class_performance: Dict = None,
               confusion_matrix: np.ndarray = None):
        for k, v in round_metrics.items():
            self.metrics.setdefault(k, []).append(v)
        for cid, acc in client_accuracies.items():
            self.client_metrics.setdefault(cid, []).append(acc)
        if class_performance:
            for cname, met in class_performance.items():
                self.class_metrics.setdefault(cname, {})
                for mname, val in met.items():
                    self.class_metrics[cname].setdefault(mname, []).append(val)
        if confusion_matrix is not None:
            self.confusion_matrices.append(confusion_matrix)

    def get_serializable_state(self):
        return {
            'metrics':            dict(self.metrics),
            'client_metrics':     dict(self.client_metrics),
            'class_metrics':      dict(self.class_metrics),
            'confusion_matrices': [cm.tolist()
                                   for cm in self.confusion_matrices]
        }

    def plot_confusion_matrix(self, class_names: List[str], final: bool = True):
        if not self.confusion_matrices:
            return
        cm   = self.confusion_matrices[-1] if final \
               else np.mean(self.confusion_matrices, axis=0)
        used = class_names[:cm.shape[0]]
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=used, yticklabels=used)
        plt.title(f'{self.n_shot}-shot '
                  f'{"Final" if final else "Average"} Confusion Matrix')
        plt.xlabel('Predicted'); plt.ylabel('True')
        plt.tight_layout()
        plt.savefig(os.path.join(
            self.results_dir,
            f'confusion_matrix_{"final" if final else "avg"}'
            f'_{self.n_shot}shot.png'))
        plt.close()

    def plot_training_curves(self):
        rounds = range(1, len(next(iter(self.metrics.values()))) + 1)

        plt.figure(figsize=(10, 6))
        plt.plot(rounds, self.metrics['accuracy'],
                 'b-', label='Global Accuracy', linewidth=2)
        plt.title(f'{self.n_shot}-shot Global Training Accuracy')
        plt.xlabel('Round'); plt.ylabel('Accuracy (%)')
        plt.grid(True, linestyle='--', alpha=0.7); plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(
            self.results_dir, f'global_accuracy_{self.n_shot}shot.png'))
        plt.close()

        plt.figure(figsize=(10, 6))
        for cid, accs in self.client_metrics.items():
            if accs:
                plt.plot(range(1, len(accs) + 1), accs,
                         marker='o', markersize=4,
                         label=f'Client {cid + 1}', linewidth=2)
        plt.title(f'{self.n_shot}-shot Client Accuracies')
        plt.xlabel('Round'); plt.ylabel('Accuracy (%)')
        plt.grid(True, linestyle='--', alpha=0.7); plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(
            self.results_dir, f'client_accuracies_{self.n_shot}shot.png'))
        plt.close()

        plt.figure(figsize=(10, 6))
        plt.plot(rounds, self.metrics['loss'],
                 'r-', label='Global Loss', linewidth=2)
        plt.title(f'{self.n_shot}-shot Global Training Loss')
        plt.xlabel('Round'); plt.ylabel('Loss')
        plt.grid(True, linestyle='--', alpha=0.7); plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(
            self.results_dir, f'global_loss_{self.n_shot}shot.png'))
        plt.close()


# ════════════════════════════════════════════════════════════════════════════
# Deviation score plot (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

def plot_deviation_scores(server: KrumServer, save_dir: str, n_shot: int):
    if not server.deviation_history:
        return
    rounds = range(1, len(server.deviation_history) + 1)
    plt.figure(figsize=(10, 6))
    for cid in range(Config.NUM_CLIENTS):
        key    = f'client_{cid}'
        scores = [r.get(key, 0.0) for r in server.deviation_history]
        if cid == BackdoorConfig.POISONED_CLIENT_ID:
            plt.plot(rounds, scores, 'r-', marker='x', linewidth=2,
                     markersize=8, label=f'Client {cid + 1} (malicious)')
        else:
            plt.plot(rounds, scores, marker='o', linewidth=1.5,
                     markersize=5, label=f'Client {cid + 1}')
    plt.title(f'{n_shot}-shot Prototype Cosine Deviation per Round')
    plt.xlabel('Round')
    plt.ylabel('Cosine Deviation Score (higher = more suspicious)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(
        save_dir, f'proto_deviation_scores_{n_shot}shot.png'))
    plt.close()


# ════════════════════════════════════════════════════════════════════════════
# Image loading / alignment (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

def load_images(directory: str, target_size: Tuple[int, int],
                is_api: bool, stats: DataStats) -> Tuple:
    images, labels, image_paths = [], [], []
    label_map    = {}
    dataset_type = 'API' if is_api else 'Traffic'

    if not os.path.exists(directory):
        raise FileNotFoundError(f"Directory not found: {directory}")

    class_names = sorted([d for d in os.listdir(directory)
                          if os.path.isdir(os.path.join(directory, d))])
    print(f"\nLoading {dataset_type} images from {directory}")

    valid_ext = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
    for label, class_name in enumerate(class_names):
        label_map[class_name] = label
        class_dir   = os.path.join(directory, class_name)
        class_count = 0
        for fname in os.listdir(class_dir):
            if os.path.splitext(fname)[1].lower() not in valid_ext:
                continue
            path = os.path.join(class_dir, fname)
            try:
                with Image.open(path) as img:
                    img   = img.convert('RGB' if is_api else 'L')
                    img   = img.resize(target_size, Image.LANCZOS)
                    arr   = np.array(img)
                    arr   = arr.transpose(2, 0, 1) if is_api else arr[None, ...]
                    arr   = (arr / 127.5) - 1.0
                    images.append(arr)
                    labels.append(label)
                    image_paths.append(path)
                    class_count += 1
            except Exception as e:
                print(f"  Skipping {path}: {e}")
        stats.add_samples(dataset_type, class_name, class_count)

    if not images:
        raise ValueError(f"No valid images loaded from {directory}")
    return (np.array(images), np.array(labels),
            class_names, label_map, image_paths)


def align_traffic_with_api(api_data, traffic_data):
    api_images, api_labels, api_paths             = api_data[:3]
    traffic_images, traffic_labels, traffic_paths = traffic_data[:3]

    api_by_label: Dict[int, List] = {}
    for img, lbl, path in zip(api_images, api_labels, api_paths):
        api_by_label.setdefault(int(lbl), []).append((img, path))

    out = {'api':     {'images': [], 'labels': [], 'paths': []},
           'traffic': {'images': [], 'labels': [], 'paths': []}}

    skipped = 0
    for t_img, lbl, t_path in zip(traffic_images, traffic_labels, traffic_paths):
        lbl = int(lbl)
        if lbl in api_by_label and api_by_label[lbl]:
            a_img, a_path = random.choice(api_by_label[lbl])
            out['api']['images'].append(a_img)
            out['api']['labels'].append(lbl)
            out['api']['paths'].append(a_path)
            out['traffic']['images'].append(t_img)
            out['traffic']['labels'].append(lbl)
            out['traffic']['paths'].append(t_path)
        else:
            skipped += 1

    if skipped:
        print(f"  [align] Skipped {skipped} traffic images "
              f"(no matching API class found)")
    print(f"  [align] Produced {len(out['api']['images'])} pairs")

    return (np.array(out['api']['images']), np.array(out['api']['labels']),
            out['api']['paths'],
            np.array(out['traffic']['images']),
            np.array(out['traffic']['labels']),
            out['traffic']['paths'])


# ════════════════════════════════════════════════════════════════════════════
# Reporting helpers (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

def save_experiment_config(config: Dict, results_dir: str):
    with open(os.path.join(results_dir, 'experiment_config.txt'), 'w') as f:
        f.write("ProtoKrum Defense Experiment Configuration\n")
        f.write("=" * 60 + "\n\n")
        for k, v in config.items():
            if isinstance(v, dict):
                f.write(f"\n{k}:\n")
                for kk, vv in v.items():
                    f.write(f"  {kk}: {vv}\n")
            else:
                f.write(f"{k}: {v}\n")


def plot_untargeted_attack_metrics(attack_results: List[Dict], save_dir: str):
    rounds = range(1, len(attack_results) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    ax1.plot(rounds, [r['attack_success_rate'] for r in attack_results],
             'r-', label='Misclassification Rate', linewidth=2)
    ax1.set_title('Backdoor Misclassification Rate over Rounds')
    ax1.set_xlabel('Round'); ax1.set_ylabel('Misclassification Rate (%)')
    ax1.grid(True, linestyle='--', alpha=0.7); ax1.legend()

    ax2.plot(rounds, [r['clean_accuracy'] for r in attack_results],
             'b-', label='Clean Accuracy', linewidth=2)
    ax2.set_title('Clean Accuracy over Rounds')
    ax2.set_xlabel('Round'); ax2.set_ylabel('Accuracy (%)')
    ax2.grid(True, linestyle='--', alpha=0.7); ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'protokrum_defense_metrics.png'))
    plt.close()

    plt.figure(figsize=(14, 8))
    for cname in attack_results[0]['class_misclassification']:
        rates = [r['class_misclassification'][cname] for r in attack_results]
        plt.plot(rounds, rates, marker='o', label=cname, linewidth=2)
    plt.title('Per-Class Misclassification Rates')
    plt.xlabel('Round'); plt.ylabel('Misclassification Rate (%)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'per_class_misclassification.png'))
    plt.close()


def save_untargeted_attack_results(attack_results: List[Dict],
                                   deviation_history: List[Dict],
                                   save_dir: str):
    path = os.path.join(save_dir, 'protokrum_defense_results.txt')
    with open(path, 'w') as f:
        f.write("ProtoKrum Defense Results\n")
        f.write("=" * 60 + "\n\n")

        final   = attack_results[-1]
        avg_asr = np.mean([r['attack_success_rate'] for r in attack_results])
        avg_ca  = np.mean([r['clean_accuracy']       for r in attack_results])

        f.write("Final Results:\n" + "-" * 20 + "\n")
        f.write(f"Clean Accuracy:              {final['clean_accuracy']:.2f}%\n")
        f.write(f"Attack Success Rate:         {final['attack_success_rate']:.2f}%\n")
        f.write(f"Clean Samples:               {final['clean_samples']}\n")
        f.write(f"Backdoor Samples:            {final['backdoor_samples']}\n\n")

        f.write("Average over all rounds:\n" + "-" * 20 + "\n")
        f.write(f"Average Clean Accuracy:      {avg_ca:.2f}%\n")
        f.write(f"Average Misclassification:   {avg_asr:.2f}%\n\n")

        f.write("Per-Class Misclassification (Final Round):\n" + "-" * 20 + "\n")
        for cname, rate in final['class_misclassification'].items():
            f.write(f"  {cname}: {rate:.2f}%\n")

        if deviation_history:
            f.write("\nPrototype Cosine Deviation Scores (per round):\n")
            f.write("-" * 20 + "\n")
            for rnd, dev in enumerate(deviation_history, 1):
                row = ", ".join(
                    f"C{i}={dev.get(f'client_{i}', 0.0):.4f}"
                    for i in range(Config.NUM_CLIENTS))
                f.write(f"  Round {rnd:02d}: {row}\n")

        f.write("\nRound-by-Round Results:\n" + "-" * 20 + "\n")
        for i, m in enumerate(attack_results, 1):
            f.write(f"\nRound {i}:\n")
            f.write(f"  Clean Accuracy:       {m['clean_accuracy']:.2f}%\n")
            f.write(f"  Misclassification:    {m['attack_success_rate']:.2f}%\n")


# ════════════════════════════════════════════════════════════════════════════
# Training loop (unchanged from Doc 8)
# ════════════════════════════════════════════════════════════════════════════

def train_untargeted_backdoor_model(server: KrumServer,
                                    num_rounds: int,
                                    local_epochs: int,
                                    results_dir: str,
                                    n_shot: int,
                                    test_dataset: Dataset,
                                    backdoor_cfg: BackdoorConfig) -> Dict:
    metrics_tracker = EnhancedMetricsTracker(results_dir, n_shot)
    os.makedirs(results_dir, exist_ok=True)

    print(f"\nStarting ProtoKrum Defense Federated Training:")
    print(f"N-shot: {n_shot}, N-way: {Config.N_WAY}, Query: {Config.N_QUERY}")
    print(f"Poisoning rate: {backdoor_cfg.POISONING_RATE}")
    print("=" * 80)

    attack_results = []
    for round_num in range(num_rounds):
        client_updates   = []
        round_losses     = []
        round_accuracies = {}

        for client in server.clients:
            update = client.train(server.global_model, local_epochs)
            client_updates.append(update)
            round_losses.append(update['avg_loss'])
            round_accuracies[client.client_id] = update['avg_accuracy']

        server.aggregate_models(client_updates)

        round_metrics = {
            'loss':     float(np.mean(round_losses)),
            'accuracy': float(np.mean(list(round_accuracies.values())))
        }

        attack_metrics = server.evaluate_untargeted_attack(
            test_dataset, backdoor_cfg)
        attack_results.append(attack_metrics)

        metrics_tracker.update(
            round_metrics     = round_metrics,
            client_accuracies = round_accuracies,
            confusion_matrix  = attack_metrics['confusion_matrix']
        )

        print(f"Round {round_num + 1}/{num_rounds}: "
              f"Loss={round_metrics['loss']:.4f}  "
              f"TrainAcc={round_metrics['accuracy']:.2f}%  "
              f"CleanAcc={attack_metrics['clean_accuracy']:.2f}%  "
              f"ASR={attack_metrics['attack_success_rate']:.2f}%")

    metrics_tracker.plot_training_curves()
    metrics_tracker.plot_confusion_matrix(test_dataset.class_names, final=True)
    plot_untargeted_attack_metrics(attack_results, results_dir)
    plot_deviation_scores(server, results_dir, n_shot)
    save_untargeted_attack_results(
        attack_results, server.deviation_history, results_dir)

    return {
        'training_metrics':  metrics_tracker.get_serializable_state(),
        'attack_results':    attack_results,
        'deviation_history': server.deviation_history,
    }


# ════════════════════════════════════════════════════════════════════════════
# Main  — FIX 1 (64x64) + FIX 2 (IID partition) applied
# ════════════════════════════════════════════════════════════════════════════

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.manual_seed(42); random.seed(42); np.random.seed(42)
    print(f"Using device: {device}")
    print(f"Image size: API={Config.API_IMAGE_SIZE}  "
          f"Traffic={Config.TRAFFIC_IMAGE_SIZE}")

    results_root = Config.RESULTS_DIR
    os.makedirs(results_root, exist_ok=True)

    experiment_config = {
        'num_clients':        Config.NUM_CLIENTS,
        'num_rounds':         Config.NUM_ROUNDS,
        'local_epochs':       Config.LOCAL_EPOCHS,
        'n_way':              Config.N_WAY,
        'n_query':            Config.N_QUERY,
        'embedding_dim':      Config.EMBEDDING_DIM,
        'episodes_per_epoch': Config.EPISODES_PER_EPOCH,
        'api_image_size':     str(Config.API_IMAGE_SIZE),
        'traffic_image_size': str(Config.TRAFFIC_IMAGE_SIZE),
        'device':             str(device),
        'poisoning_rates':    str(POISONING_RATES),
        'trigger_size':       BackdoorConfig.TRIGGER_SIZE,
        'train_test_split':   '80% train / 20% test (stratified)',
        'client_partition':   'IID — proportional share per class',
        'defense_method': {
            'method':      'ProtoKrum (Cosine Deviation + Weighted Krum)',
            'description': 'Stage 1: cosine deviation pre-weighting; '
                           'Stage 2: amplified Krum parameter scoring',
            'num_adversaries': 1
        },
        'backdoor_config': {
            'type':            'Untargeted',
            'poisoned_client': BackdoorConfig.POISONED_CLIENT_ID,
            'trigger_size':    BackdoorConfig.TRIGGER_SIZE,
            'scale_factor':    BackdoorConfig.SCALE_FACTOR
        }
    }
    save_experiment_config(experiment_config, results_root)

    try:
        data_stats = DataStats()
        print("\nLoading and preprocessing data...")
        api_images, api_labels, api_classes, _, api_paths = load_images(
            Config.API_IMAGE_DIR, Config.API_IMAGE_SIZE, True, data_stats)
        traffic_images, traffic_labels, _, _, traffic_paths = load_images(
            Config.TRAFFIC_IMAGE_DIR, Config.TRAFFIC_IMAGE_SIZE, False, data_stats)

        data_stats.display_distribution()
        data_stats.plot_distribution(
            os.path.join(results_root, 'data_distribution.png'))

        print("\nAligning traffic data with API data...")
        (api_images, api_labels, api_paths,
         traffic_images, traffic_labels, traffic_paths) = align_traffic_with_api(
            (api_images, api_labels, api_paths),
            (traffic_images, traffic_labels, traffic_paths))

        # ── Stratified 80/20 split — applied once, outside all loops ───────
        print("\nSplitting data into 80% train / 20% test...")
        (tr_api, tr_trf, tr_lbl,
         te_api, te_trf, te_lbl) = split_data(
            api_images, traffic_images, api_labels,
            test_ratio=0.2, seed=42)
        print(f"Train: {len(tr_lbl)} | Test: {len(te_lbl)} samples")

        all_results = {}
        for poison_rate in POISONING_RATES:
            print(f"\n{'#'*20}  Poisoning rate = {poison_rate}  {'#'*20}")

            bdcfg = BackdoorConfig()
            bdcfg.POISONING_RATE = poison_rate

            rate_tag = f"rate{int(poison_rate * 100)}"
            rate_dir = os.path.join(results_root, rate_tag)
            os.makedirs(rate_dir, exist_ok=True)

            rate_results = {}
            for n_shot in [1, 5]:
                print(f"\n{'='*20} {n_shot}-shot | rate={poison_rate} {'='*20}")
                run_dir = os.path.join(rate_dir, f'{n_shot}shot_protokrum')
                os.makedirs(run_dir, exist_ok=True)

                # ── FIX 2: IID partition of training data ─────────────────
                partitions = iid_partition(
                    tr_api, tr_trf, tr_lbl, Config.NUM_CLIENTS)
                print_client_distribution(partitions, api_classes)

                # Per-client training datasets (training split only)
                client_datasets = [
                    UntargetedBackdoorDataset(
                        p['api'], p['traffic'],
                        p['labels'].copy(), api_classes, n_shot)
                    for p in partitions
                ]

                # Held-out test dataset (test split only, never seen in training)
                test_dataset = UntargetedBackdoorDataset(
                    te_api, te_trf,
                    te_lbl.copy(), api_classes, n_shot)

                model  = HybridNet().to(device)
                server = create_krum_backdoor_system(
                    model           = model,
                    datasets        = client_datasets,  # FIX 2: not [ds]*5
                    device          = device,
                    backdoor_config = bdcfg,
                    num_adversaries = 1
                )

                training_results = train_untargeted_backdoor_model(
                    server       = server,
                    num_rounds   = Config.NUM_ROUNDS,
                    local_epochs = Config.LOCAL_EPOCHS,
                    results_dir  = run_dir,
                    n_shot       = n_shot,
                    test_dataset = test_dataset,
                    backdoor_cfg = bdcfg
                )
                rate_results[n_shot] = training_results

            all_results[poison_rate] = rate_results

        # ── Summary table ─────────────────────────────────────────────────
        comparison_path = os.path.join(results_root, 'final_protokrum_results.txt')
        with open(comparison_path, 'w') as f:
            f.write("ProtoKrum Defense Results Summary\n")
            f.write("=" * 60 + "\n\n")
            f.write("Defense: Prototype Cosine Deviation + Weighted Krum\n")
            f.write(f"Image size: {Config.API_IMAGE_SIZE}\n")
            f.write("Data split: 80% train / 20% test (stratified)\n")
            f.write("Client partition: IID proportional per class\n")
            f.write(f"Clients: {Config.NUM_CLIENTS}  |  "
                    f"Poisoned: {BackdoorConfig.POISONED_CLIENT_ID}  |  "
                    f"Trigger: {BackdoorConfig.TRIGGER_SIZE}\n\n")

            f.write(f"{'PR':<6} {'Shot':<6} {'ACC (final)':<15} "
                    f"{'ASR (final)':<15} {'Avg ACC':<12} {'Avg ASR':<12}\n")
            f.write("-" * 70 + "\n")

            for poison_rate in POISONING_RATES:
                for n_shot in [1, 5]:
                    ar      = all_results[poison_rate][n_shot]['attack_results']
                    last    = ar[-1]
                    avg_ca  = np.mean([r['clean_accuracy']      for r in ar])
                    avg_asr = np.mean([r['attack_success_rate'] for r in ar])
                    f.write(f"{poison_rate:<6} {n_shot:<6} "
                            f"{last['clean_accuracy']:<15.2f} "
                            f"{last['attack_success_rate']:<15.2f} "
                            f"{avg_ca:<12.2f} {avg_asr:<12.2f}\n")

        # Print final table to console
        print("\n" + "="*60)
        print("FINAL RESULTS")
        print("="*60)
        print(f"{'PR':<6} {'Shot':<6} {'ACC%':<10} {'ASR%':<10}")
        for poison_rate in POISONING_RATES:
            for n_shot in [1, 5]:
                last = all_results[poison_rate][n_shot]['attack_results'][-1]
                print(f"{poison_rate:<6} {n_shot:<6} "
                      f"{last['clean_accuracy']:<10.2f} "
                      f"{last['attack_success_rate']:<10.2f}")

        print(f"\nExperiment completed. Results in: {results_root}")

    except Exception as e:
        print(f"\nError: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()


def run_krum_defense_experiment():
    print("=" * 60)
    print("ProtoKrum Defense Against Untargeted Backdoor Attacks")
    print("=" * 60)
    try:
        os.makedirs(Config.RESULTS_DIR, exist_ok=True)
        main()
        print("\nExperiment Completed Successfully!")
        print(f"Results: {Config.RESULTS_DIR}")
    except Exception as e:
        print(f"Experiment failed: {e}")
        import traceback
        traceback.print_exc()


if __name__ == '__main__':
    run_krum_defense_experiment()

ProtoKrum Defense Against Untargeted Backdoor Attacks
Using device: cuda
Image size: API=(64, 64)  Traffic=(64, 64)

Loading and preprocessing data...

Loading API images from H:\CSIE\Fall 2024\Cyber Security threat analysis\Assignments\Final Project\API_Images

Loading Traffic images from H:\CSIE\Fall 2024\Cyber Security threat analysis\Assignments\Final Project\Tspilit_image

Overall Data Distribution:
          API  Traffic  Total
Cryptbot   14       44     58
Formbook   15      268    283
Remcos     15      156    171
Rokrat     14      129    143
Stealc     15       88    103
Vidar      15       88    103
Xworm      14       13     27

Total Samples: 888

Aligning traffic data with API data...
  [align] Produced 786 pairs

Splitting data into 80% train / 20% test...

[split_data] Total=786  Train=628 (80%)  Test=158 (20%)  [stratified]
  [split_data] Test class counts: cls0=9, cls1=54, cls2=31, cls3=26, cls4=18, cls5=18, cls6=2
  [Warning] Smallest test class has 2 sample(s).
Trai